In [ ]:
import os
import requests
import json
import torch

from pathlib import Path
from datasets import load_dataset



In [ ]:

import sys
sys.path.append("..")


from src.tokenizer import ScratchyTokenizer

In [ ]:
data_path = Path("../data")
ckpt_path = Path("../ckpt")

ckpt_path.mkdir(exist_ok=True, parents=True)
warandpeace_path = data_path / "warandpeace.txt"


tokenizer_path = ckpt_path / 'tokenizer'
tokenizer_path.mkdir(exist_ok=True, parents=True)

In [ ]:
with open(warandpeace_path, 'r') as file:
    warandpeace = file.read()

In [ ]:
tokenizer_scratchy_path = tokenizer_path / "scratchy.json"

with open(tokenizer_scratchy_path, 'r') as file:
    chars = json.load(file)

In [ ]:

tokenizer = ScratchyTokenizer(chars)
data = torch.tensor(tokenizer.encode(warandpeace), dtype=torch.long)
data.shape

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

In [ ]:

torch.manual_seed(1337)

In [ ]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [ ]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
class BigramLM(torch.nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)

            logits = logits[:, -1, :] # (B, C)

            probs = F.softmax(logits, dim=-1) # (B, C)

            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            idx = torch.cat([idx, idx_next], dim=1) # (B, C + 1)
        return idx
    

In [ ]:

model = BigramLM(tokenizer.vocab_size)
m = model.to(device)

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


In [ ]:

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step: {iter}: train_loss {losses["train"]:.4f}, val_loss {losses["val"]:.4f}")

    inputs, targets = get_batch("train")

    logits, loss = model(inputs, targets)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


In [ ]:

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(tokenizer.decode(m.generate(context, max_new_tokens=500)[0].tolist()))